# AI 기술 면접관 변환기 (LoRA, 양자화 없음)

이 노트북은 **Qwen/Qwen3-1.7B** 모델을 양자화하지 않고 `bf16`으로 불러온 뒤, **LoRA 어댑터만 학습**해서 지원자 경험을 바탕으로 심층 기술면접 질문을 생성하는 실습입니다.

핵심 흐름은 다음과 같습니다.

1. 파인튜닝 전 기본 모델의 답변을 확인합니다.
2. 지원자 경험(`instruction`+`input`) → 기술면접 질문(`output`) 쌍 31개를 `prompt` / `completion` 형태로 만듭니다.
3. 원본 모델 가중치는 고정하고 LoRA 어댑터만 학습합니다.
4. 같은 질문에 대해 파인튜닝 전/후 답변을 비교합니다.
5. 학습된 LoRA 어댑터만 파일로 저장합니다.

> 이 노트북은 **QLoRA가 아닙니다.** 모델 가중치를 4bit로 불러오지 않습니다. 다만 학습 시 VRAM을 아끼기 위해 optimizer는 `paged_adamw_8bit`를 사용합니다.


In [1]:
import os
import warnings
import logging

# 1. 파이썬 기본 경고 무시
warnings.filterwarnings("ignore")

# 2. 시스템 환경변수를 통한 Hugging Face 및 커널 로그 제어 (0=ALL, 1=INFO, 2=WARNING, 3=ERROR)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["LOGGERS_LEVEL"] = "ERROR"

# 3. transformers 자체 라이브러리 로그 레벨을 ERROR로 설정
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

## 0. 환경 확인

먼저 Python, PyTorch, CUDA, GPU, VRAM 상태를 확인합니다. 이 실습은 **Python 3.12 + JupyterLab** 환경을 기준으로 작성되었습니다.

- NVIDIA GPU가 정상적으로 인식되어야 합니다.
- 기본 설정은 8GB VRAM급 GPU에서도 실행되도록 보수적으로 잡았습니다.
- VRAM이 부족하면 뒤쪽 학습 설정에서 `max_length`를 줄이거나 `gradient_accumulation_steps`를 조정하세요.


In [2]:
import torch, platform

print(f"Python 버전: {platform.python_version()}")
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"총 VRAM: {total_vram:.1f} GB")
else:
    print("⚠️ GPU가 감지되지 않았습니다. NVIDIA 드라이버 / CUDA 설치를 확인하세요.")


Python 버전: 3.12.13
PyTorch 버전: 2.10.0+cu128
CUDA 사용 가능: True
GPU 이름: NVIDIA GeForce RTX 3060
총 VRAM: 12.0 GB


## 1. 패키지 설치 및 버전 확인

처음 실행하는 환경이라면 필요한 패키지를 한 번만 설치합니다. 이미 설치되어 있다면 설치 셀은 건너뛰고, 바로 버전 확인 셀만 실행해도 됩니다.

이 노트북에서 사용하는 주요 패키지는 다음과 같습니다.

- `transformers`: Qwen 모델과 tokenizer 로드
- `peft`: LoRA 어댑터 구성 및 저장
- `trl`: `SFTTrainer`를 이용한 지도 미세조정
- `bitsandbytes`: `paged_adamw_8bit` optimizer 사용
- `datasets`: 학습 데이터셋 구성

> 여기서 `bitsandbytes`를 쓰지만, 이 노트북은 모델을 4bit로 양자화하지 않습니다. `bitsandbytes`는 optimizer 메모리 절약을 위해 사용됩니다.


In [3]:
# 최초 1회만 실행하세요. (앞의 # 을 지우고 실행)
# uv add "transformers>=5.10.1" "peft>=0.19.0" "trl>=0.24.0" "bitsandbytes>=0.48.0" \
#     "accelerate>=1.11.0" "datasets>=3.0.0" sentencepiece protobuf -U


In [4]:
import datasets

In [5]:
import transformers, peft, trl, bitsandbytes, accelerate

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [6]:
print("transformers :", transformers.__version__)
print("peft         :", peft.__version__)
print("trl          :", trl.__version__)
print("bitsandbytes :", bitsandbytes.__version__)
print("accelerate   :", accelerate.__version__)
print("datasets     :", datasets.__version__)


transformers : 5.5.0
peft         : 0.19.1
trl          : 0.24.0
bitsandbytes : 0.49.2
accelerate   : 1.14.0
datasets     : 4.3.0


## 2. 모델 로드 (양자화 없음 · bf16)

기본 모델은 **`Qwen/Qwen3-1.7B`** 입니다. 이 셀에서는 `BitsAndBytesConfig` 없이 모델을 그대로 `bf16`으로 GPU에 올립니다.

코드에서 중요한 부분은 다음과 같습니다.

- `torch_dtype=torch.bfloat16`: 모델을 bf16 정밀도로 로드합니다.
- `device_map={"": 0}`: 단일 GPU 0번에 모델을 명시적으로 올립니다.
- `tokenizer.pad_token`이 없으면 `eos_token`으로 대체합니다.

이번 실습의 목적은 양자화 기법이 아니라 **LoRA 어댑터가 답변 스타일을 어떻게 바꾸는지** 확인하는 것입니다.


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
)

print("모델 로드 완료! (양자화 없음, bf16)")
if torch.cuda.is_available():
    print(f"현재 GPU 메모리 사용량: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

모델 로드 완료! (양자화 없음, bf16)
현재 GPU 메모리 사용량: 3.20 GB


## 3. 응답 생성 함수

`ask()` 함수는 같은 질문을 파인튜닝 전과 후에 반복해서 테스트하기 위한 헬퍼 함수입니다.

- `tokenizer.apply_chat_template()`으로 Qwen 채팅 형식에 맞는 입력 문장을 만듭니다.
- `enable_thinking=False`로 thinking 모드가 아니라 일반 답변 모드로 생성합니다.
- `model.eval()`을 호출해 생성 중 LoRA dropout이 켜지지 않도록 합니다.
- `torch.no_grad()`로 추론 중 gradient 계산을 끕니다.


In [28]:
def ask(q, max_new_tokens=200):
    model.eval()  # LoRA dropout이 켜진 채 생성하면 답변이 불안정해지므로 항상 eval 모드로 고정
     # Qwen3의 chat template에 맞는 message 구조
    # role은 system / user / assistant 형태를 사용한다.
    SYSTEM_PROMPT = """당신은 AI/LLM 개발자 채용을 담당하는 전문 기술면접관이다.
    지원자의 이력과 기술 경험을 분석하여 실제 역량을 검증할 수 있는
    구체적이고 심층적인 기술 질문을 생성한다.
    단순 정의 암기보다는 설계 이유, 트레이드오프, 문제 해결, 평가 방법을 질문한다."""
    
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                "다음 지원자의 실제 기술 역량을 검증할 수 있는 "
                "심층 기술면접 질문 3개를 만들어라.\n\n"
                + q
            ),
        },
    ]
    text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.5,
            top_p=0.8,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
        )
    print(output.shape)
    response = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    return response.strip()


## 4. Before: 파인튜닝 전 응답 확인

학습 전 모델이 같은 질문에 어떻게 답하는지 확인합니다. 이 결과는 `before_answers`에 저장해 두고, 학습 후 답변과 나란히 비교합니다.


In [9]:
torch.cuda.is_bf16_supported()

True

In [ ]:
test_questions = ["직무: RAG 개발자\n경험: LangChain, Qdrant, BGE 임베딩을 사용해 사내 문서 QA 챗봇을 개발했다. Top-k=5를 사용했다.",
"직무: LLM 엔지니어\n경험: Qwen 계열 모델을 LoRA로 파인튜닝했고 PEFT와 Transformers를 사용했다.", 
"직무: LLM 엔지니어\n경험: 4-bit NF4와 QLoRA를 사용해 8GB GPU에서 모델을 파인튜닝했다."]

before_answers = {}
print("=" * 60)
print("파인튜닝 전(Before) 응답")
print("=" * 60)
for q in test_questions:
    ans = ask(q)
    before_answers[q] = ans
    # print(f"\nQ: {q}\nA: {ans}")
    print(f"\nQ: {q}\n\nA: {ans}")
    print("-" * 50)


파인튜닝 전(Before) 응답
torch.Size([1, 184])

Q: 직무: RAG 개발자
경험: LangChain, Qdrant, BGE 임베딩을 사용해 사내 문서 QA 챗봇을 개발했다. Top-k=5를 사용했다.

A: 1. 질문이 없는 경우도 있으니 반드시 질문만 확인하고 답변을 줘야 합니다.
2. 단순한 사실보다 학습 데이터와 모델의 관계나 예측 오류 원인 등을 설명해보세요.
3. 다양한 검색 조건과 관련된 범주를 구분해서 정확도가 낮은 이유를 분석해보겠습니다.
4. top-k=10으로 바꾸면 어떤 효과가 있을까요?
5. 추가적인 요약 정보를 넣어질 때 어떤 기준을 적용하겠습니까?
--------------------------------------------------
torch.Size([1, 158])

Q: 직무: LLM 엔지니어
경험: Qwen 계열 모델을 LoRA로 파인튜닝했고 PEFT와 Transformers를 사용했다.

A: 1. LoRA의 학습 효과가 낮은 경우 어떤 이유를 의심하겠습니까?
2. 특정 layer만 fine-tuning할 때 어떤 문제가 생길 수 있나요?
3. optimizer 설정이 과적합으로 인해 일반화력 저하되었으면서도 validation loss가 줄어드는 현상은 무엇일까요?
4. batch size가 너무 작아서 training time이 길어졌다면 어떤 조정을 시도해볼 수 있습니까?
--------------------------------------------------
torch.Size([1, 145])

Q: 직무: LLM 엔지니어
경험: 4-bit NF4와 QLoRA를 사용해 8GB GPU에서 모델을 파인튜닝했다.

A: 1. 학습 데이터의 크기를 줄이기 위해 어떤 방법을 사용했나요?
2. model size과 batch size가 증가할 때 성능 저하를 어떻게 관리하겠습니까?
3. training loss가 낮아질 때 optimizer 설정에 어떤 조정을 시도했나 봐요?
4. gradient 

## 5. 데이터셋 준비

지원자 경험 설명과 기술면접 질문 쌍 31개를 학습 데이터로 사용합니다. 다음 셀에서는 JSONL로 불러온 데이터를 Hugging Face `Dataset`으로 바꾼 뒤, 각 예제를 `prompt`와 `completion`으로 변환합니다.

- `prompt`: 사용자 질문과 assistant 시작 토큰까지 포함한 입력 부분
- `completion`: 모델이 학습해야 하는 기술면접 질문(답변) 부분

뒤쪽 `SFTConfig`에서 `completion_only_loss=True`를 사용하므로, loss는 주로 `completion` 부분에만 계산됩니다. 즉, 모델은 지원자 경험 문장을 외우기보다 **면접관으로서 질문을 만드는 방식**을 배우게 됩니다.


In [11]:
# Hugging Face load_dataset 사용
from datasets import load_dataset

# 데이터 불러오기
data_path = "datas/ai_interview_sft.jsonl"

if data_path:
    # JSONL 파일을 바로 Hugging Face Dataset으로 불러오기
    raw_data = load_dataset("json", data_files=str(data_path), split="train")

    print(raw_data)
    print("샘플 수:", len(raw_data))
    print(raw_data[0])


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 31
})
샘플 수: 31
{'instruction': '지원자의 경험을 바탕으로 실제 역량을 검증할 수 있는 심층 기술면접 질문 3개를 만들어라.', 'input': '직무: RAG 개발자\n경험: LangChain, Qdrant, BGE 임베딩을 사용해 사내 문서 QA 챗봇을 개발했다. Top-k=5를 사용했다.', 'output': '1. Qdrant를 선택한 이유를 FAISS와 비교하여 설명해보세요.\n2. Top-k=5를 결정할 때 어떤 실험이나 평가 지표를 사용했나요?\n3. 임베딩 검색 결과는 관련성이 높지만 최종 답변이 부정확할 때 어느 단계를 우선 점검하겠습니까?'}


In [12]:
from datasets import Dataset

# raw_data = [{"instruction": q, "response": a} for q, a in dataset]

def format_example(example):
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": example["instruction"] + example["input"]}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    completion = example["output"] + tokenizer.eos_token
    return {"prompt": prompt, "completion": completion}

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_example)
print("PROMPT:", dataset[0]["prompt"])
print("COMPLETION:", dataset[0]["completion"])


Map:   0%|          | 0/31 [00:00<?, ? examples/s]

PROMPT: <|im_start|>user
지원자의 경험을 바탕으로 실제 역량을 검증할 수 있는 심층 기술면접 질문 3개를 만들어라.직무: RAG 개발자
경험: LangChain, Qdrant, BGE 임베딩을 사용해 사내 문서 QA 챗봇을 개발했다. Top-k=5를 사용했다.<|im_end|>
<|im_start|>assistant
<think>

</think>


COMPLETION: 1. Qdrant를 선택한 이유를 FAISS와 비교하여 설명해보세요.
2. Top-k=5를 결정할 때 어떤 실험이나 평가 지표를 사용했나요?
3. 임베딩 검색 결과는 관련성이 높지만 최종 답변이 부정확할 때 어느 단계를 우선 점검하겠습니까?<|im_end|>


## 6. LoRA 어댑터 설정 (양자화 없음)

이 노트북은 QLoRA가 아니므로 `prepare_model_for_kbit_training()`을 사용하지 않습니다. 대신 다음 순서로 순정 LoRA 학습을 준비합니다.

1. `model.gradient_checkpointing_enable()`로 메모리 사용량을 줄입니다.
2. `model.enable_input_require_grads()`로 gradient checkpointing 환경에서 입력 gradient를 허용합니다.
3. `LoraConfig`로 어댑터 설정을 정의합니다.
4. `get_peft_model()`로 원본 모델에 LoRA 어댑터를 붙입니다.

| 파라미터 | 의미 | 이번 실습 값 |
|---|---|---|
| `r` | LoRA rank. 클수록 표현력과 메모리 사용량이 증가합니다. | 16 |
| `lora_alpha` | LoRA scaling 계수입니다. | 32 |
| `lora_dropout` | 과적합 방지용 dropout입니다. | 0.05 |
| `target_modules` | LoRA를 붙일 attention/MLP projection layer입니다. | `q/k/v/o`, `gate/up/down` |

`model.print_trainable_parameters()` 결과에서 학습 가능한 파라미터가 0보다 커야 정상입니다. 0으로 나오면 LoRA 어댑터가 학습 대상이 아니므로, 모델 로드 셀부터 이 셀까지 순서대로 다시 실행하세요.


In [13]:
from peft import LoraConfig, get_peft_model

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
# y = W x + (α / r) B A x
# W x              = 원본 모델의 출력
# (α / r) B A x    = LoRA adapter가 만든 보정 출력
# α / r은 LoRA 보정 출력이 원본 출력에 얼마나 강하게 반영될지 조절하는 값
# scale = (α / r)
# r : r = adapter 크기, 표현력, 병목 차원
# α : LoRA adapter의 영향력 크기를 조절하는 하이퍼파라미터
# α는 adapter의 학습 파라미터 수를 늘리는 값이 아니다. 오직 보정값의 세기를 조절한다.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


In [14]:
# r = 8
# trainable params: 8,716,288 || all params: 1,729,291,264 || trainable%: 0.5040

# r=16
# trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030

# r=32
# trainable params: 34,865,152 || all params: 1,755,440,128 || trainable%: 1.9861

## 7. 학습 설정 및 실행

`SFTTrainer`로 기술면접 질문 생성을 지도 미세조정합니다. 이 실습은 작은 데이터셋으로 질문 스타일 변화를 확인하는 것이 목적이므로, 전체 모델이 아니라 LoRA 어댑터만 업데이트합니다.

주요 설정은 다음과 같습니다.

- `per_device_train_batch_size=1`: GPU 메모리를 아끼기 위해 실제 배치는 1로 둡니다.
- `gradient_accumulation_steps=8`: 8번의 mini-batch를 모아 한 번 업데이트합니다.
- `learning_rate=2e-4`: LoRA 학습에서 자주 쓰는 범위의 학습률입니다.
- `bf16=True`: bf16 연산을 사용합니다.
- `optim="paged_adamw_8bit"`: optimizer 상태 메모리를 절약합니다.
- `max_length=512`: 입력과 출력 토큰을 합친 최대 길이입니다.
- `completion_only_loss=True`: 질문이 아니라 assistant 답변 부분 위주로 loss를 계산합니다.

학습이 끝나면 `model.eval()`로 전환합니다. LoRA dropout이 켜진 train mode에서 생성하면 답변이 불안정해질 수 있기 때문입니다.

> 오류 점검: loss가 전혀 줄지 않거나 학습이 되는 것처럼 보이는데 결과가 변하지 않으면, `model.print_trainable_parameters()`에서 trainable parameter가 0이 아닌지 먼저 확인하세요.


In [15]:
from trl import SFTTrainer, SFTConfig

# SFT을 config 설정 객체 생성
training_args = SFTConfig(
    output_dir="./output/sample_a1_interviewer_lora",
    num_train_epochs=8,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    bf16=True,
    optim="paged_adamw_8bit",
    max_length=512,
    completion_only_loss=True,
    report_to="none",
)

# SFT LoRA 학습을 위한 객체 생성
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

# LoRA FT 학습
trainer.train()

# 학습이 끝나면 반드시 eval 모드로 전환합니다.
# (LoRA dropout이 생성 중에도 계속 켜져 있으면 답변이 불안정해집니다)
model.eval()
print("학습 완료! model.eval() 적용됨 — 이제 안정적으로 답변을 생성할 수 있습니다.")


Adding EOS to train dataset:   0%|          | 0/31 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/31 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/31 [00:00<?, ? examples/s]

{'loss': '2.556', 'grad_norm': '0.1507', 'learning_rate': '0.000175', 'entropy': '1.679', 'num_tokens': '6372', 'mean_token_accuracy': '0.553', 'epoch': '1.258'}
{'loss': '1.762', 'grad_norm': '0.1383', 'learning_rate': '0.0001437', 'entropy': '2.393', 'num_tokens': '1.273e+04', 'mean_token_accuracy': '0.6262', 'epoch': '2.516'}
{'loss': '1.497', 'grad_norm': '0.1229', 'learning_rate': '0.0001125', 'entropy': '2.296', 'num_tokens': '1.924e+04', 'mean_token_accuracy': '0.6832', 'epoch': '3.774'}
{'loss': '1.305', 'grad_norm': '0.1327', 'learning_rate': '8.125e-05', 'entropy': '2.039', 'num_tokens': '2.542e+04', 'mean_token_accuracy': '0.715', 'epoch': '5'}
{'loss': '1.132', 'grad_norm': '0.137', 'learning_rate': '5e-05', 'entropy': '1.821', 'num_tokens': '3.178e+04', 'mean_token_accuracy': '0.7455', 'epoch': '6.258'}
{'loss': '1.066', 'grad_norm': '0.1339', 'learning_rate': '1.875e-05', 'entropy': '1.766', 'num_tokens': '3.822e+04', 'mean_token_accuracy': '0.7583', 'epoch': '7.516'}
{'t

## 8. After: 파인튜닝 후 응답 비교

학습 전에 저장해 둔 `before_answers`와 학습 후 `ask()` 결과를 나란히 출력합니다. 지원자 경험을 파고드는 심층 기술면접 질문이 일관되게 생성되는지 확인합니다.


In [22]:
test_questions = ["직무: RAG 개발자\n경험: LangChain, Qdrant, BGE 임베딩을 사용해 사내 문서 QA 챗봇을 개발했다. Top-k=5를 사용했다.",
"직무: LLM 엔지니어\n경험: Qwen 계열 모델을 LoRA로 파인튜닝했고 PEFT와 Transformers를 사용했다.", 
"직무: LLM 엔지니어\n경험: 4-bit NF4와 QLoRA를 사용해 8GB GPU에서 모델을 파인튜닝했다."]

In [31]:
print("=" * 60)
print("Before vs After 비교")
print("=" * 60)
for q in test_questions:
    after_ans = ask(q)
    print(f"\nQ: {q}")
    print()
    print(f"[Before]\n{before_answers[q]}")
    print()
    print(f"[After ]\n{after_ans}")
    print("-" * 50)


Before vs After 비교
torch.Size([1, 259])

Q: 직무: RAG 개발자
경험: LangChain, Qdrant, BGE 임베딩을 사용해 사내 문서 QA 챗봇을 개발했다. Top-k=5를 사용했다.

[Before]
1. 질문이 없는 경우도 있으니 반드시 질문만 확인하고 답변을 줘야 합니다.
2. 단순한 사실보다 학습 데이터와 모델의 관계나 예측 오류 원인 등을 설명해보세요.
3. 다양한 검색 조건과 관련된 범주를 구분해서 정확도가 낮은 이유를 분석해보겠습니다.
4. top-k=10으로 바꾸면 어떤 효과가 있을까요?
5. 추가적인 요약 정보를 넣어질 때 어떤 기준을 적용하겠습니까?

[After ]
1. top-k와 nucleus sampling이 각각 어떤 상황에서 더 적합한지 설명하고 why?
2. embedding similarity threshold를 너무 낮게 설정하면 무엇이 나올 수 있나요?
3. retriever score boosting이 필요한 경우 어떤 metric을 먼저 확인하겠습니까?
--------------------------------------------------
torch.Size([1, 281])

Q: 직무: LLM 엔지니어
경험: Qwen 계열 모델을 LoRA로 파인튜닝했고 PEFT와 Transformers를 사용했다.

[Before]
1. LoRA의 학습 효과가 낮은 경우 어떤 이유를 의심하겠습니까?
2. 특정 layer만 fine-tuning할 때 어떤 문제가 생길 수 있나요?
3. optimizer 설정이 과적합으로 인해 일반화력 저하되었으면서도 validation loss가 줄어드는 현상은 무엇일까요?
4. batch size가 너무 작아서 training time이 길어졌다면 어떤 조정을 시도해볼 수 있습니까?

[After ]
1. 특정 task에 맞춘 fine-tuning hyperparameter 조정이 성능 향상에 효과적이지 않은 경우 어떤 원인을 추측하겠습니까?
2. para

## 9. LoRA 어댑터 저장

이 셀은 전체 모델이 아니라 **LoRA 어댑터만** 저장합니다. 저장 위치는 다음과 같습니다.

```text
./lora_adapters/exam4_interviewer_lora
```

어댑터만 저장하면 파일 크기가 작고, 나중에 같은 base model에 다시 붙여서 사용할 수 있습니다. 전체 모델로 배포하려면 별도의 병합 과정이 필요합니다.


In [19]:
ADAPTER_DIR = "./lora_adapters/4_interviewer_lora"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA 어댑터 저장 완료: {ADAPTER_DIR}")


LoRA 어댑터 저장 완료: ./lora_adapters/4_interviewer_lora


## 정리

이 노트북에서 확인한 내용은 다음과 같습니다.

- 양자화 없이 `Qwen/Qwen3-1.7B`를 bf16으로 로드했습니다.
- 원본 모델 전체를 학습하지 않고 LoRA 어댑터만 학습했습니다.
- `prompt` / `completion` 데이터 구조와 `completion_only_loss=True`를 사용해 면접관이 던지는 심층 기술면접 질문 생성 방식을 중심으로 학습했습니다.
- 파인튜닝 전/후 답변을 비교해 질문 품질과 스타일 변화가 실제로 일어나는지 확인했습니다.
- 최종 산출물로 LoRA 어댑터를 저장했습니다.

다음 단계로는 저장된 어댑터를 다시 불러오거나, base model과 병합한 뒤 Ollama 같은 로컬 실행 환경에 배포할 수 있습니다.


In [20]:
Trainable Ratio       : __________ %
Peak Allocated VRAM   : __________ GB
Peak Reserved VRAM    : __________ GB
Final Train Loss      : __________

학습 전 특징:
________________________________________________

학습 후 특징:
________________________________________________

SyntaxError: invalid syntax (524439272.py, line 1)